<a href="https://colab.research.google.com/github/Egorch-max/ML-2026-Frolov/blob/main/dataset_overview_Frolov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Датасет: Inside Airbnb - Bologna
**Фролов Егор, 3823Б1ПМоп2**

Источник: [insideairbnb.com](https://insideairbnb.com/get-the-data/), файл `listings.csv.gz` (Detailed Listings, 26.06.2026).

**Задача:** регрессия — предсказать цену аренды за ночь (`price`, EUR) по характеристикам объявления.

In [1]:
import pandas as pd
import numpy as np

URL = "https://data.insideairbnb.com/italy/emilia-romagna/bologna/2026-06-26/data/listings.csv.gz"
df = pd.read_csv(URL)
print(df.shape)
df.head()

(4948, 90)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,46352,https://www.airbnb.com/rooms/46352,20260626225529,2026-06-27,city scrape,A room in Pasolini's house,"Simple, cozy and silent room in a lived house ...",NaN,https://a0.muscache.com/pictures/9f94acc3-98fc...,467810,...,4.86,4.84,4.68,IT037006C1C1767,NaN,2,0,2,0,2.59
1,59697,https://www.airbnb.com/rooms/59697,20260626225529,2026-06-27,city scrape,COZY LARGE BEDROOM in the city center,"Cozy, spacious and bright double bedroom, in a...",NaN,https://a0.muscache.com/pictures/ac0528c4-b26f...,286688,...,4.80,4.83,4.66,NaN,NaN,2,0,2,0,1.47
2,85368,https://www.airbnb.com/rooms/85368,20260626225529,2026-06-27,city scrape,Garden House Bologna,NaN,NaN,https://a0.muscache.com/pictures/585815/331d45...,467675,...,4.99,4.83,4.83,IT037006C2QKQSH3S7,NaN,1,1,0,0,0.41
3,219878,https://www.airbnb.com/rooms/219878,20260626225529,2026-06-27,city scrape,Top floor with elevator for Long and Short Stays,"Bright, quiet apartment with a private office ...",NaN,https://a0.muscache.com/pictures/f16e5b7e-68b3...,1139326,...,4.93,4.85,4.78,IT037006C2HQERHOGR,NaN,1,1,0,0,1.11
4,229114,https://www.airbnb.com/rooms/229114,20260626225529,2026-06-27,city scrape,Apartment with terrace between center and fair,"Bright apartment for 4 people in Bologna, conv...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,1196205,...,4.95,4.53,4.71,IT037006C2KRRTWGRT,NaN,1,1,0,0,0.67


## 1. Отбор признаков
Убираем полностью пустые столбцы, служебные (id, ссылки, даты выгрузки), целевую переменную и столбцы, которые «подсказывают» цену (утечка).

In [2]:
target = "price"

empty = [c for c in df.columns if df[c].isna().all()]
service = ["id", "listing_url", "scrape_id", "last_scraped", "source", "picture_url",
           "host_id", "host_url", "host_profile_id", "host_profile_url", "host_picture_url",
           "host_name", "calendar_last_scraped", "license"]
leak = ["price_quote_total_price", "price_quote_price_per_night", "price_quote_raw",
        "price_quote_checkin_date", "price_quote_checkout_date", "estimated_revenue_l365d"]

features = [c for c in df.columns if c not in set(empty + service + leak + [target])]
print("Полностью пустые столбцы:", len(empty), empty)
print("N (объектов):", len(df))
print("d (признаков):", len(features))

Полностью пустые столбцы: 13 ['neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_group_cleansed', 'calendar_updated', 'instant_bookable']
N (объектов): 4948
d (признаков): 56


## 2. Типы признаков

In [3]:
text_cols  = ["name", "description", "host_about"]
cat_cols   = ["host_location", "host_is_superhost", "host_has_profile_pic", "host_identity_verified",
              "neighbourhood_cleansed", "property_type", "room_type", "bathrooms_text", "has_availability"]
other_cols = ["amenities", "first_review", "last_review"]   # список удобств и даты
num_cols   = [c for c in features if c not in text_cols + cat_cols + other_cols]

print("Числовых:", len(num_cols))
print("Категориальных:", len(cat_cols))
print("Текстовых:", len(text_cols))
print("Других типов:", len(other_cols), other_cols)
print()
print("Пример текстового признака (description):")
print(df["description"].dropna().iloc[0][:400])

Числовых: 41
Категориальных: 9
Текстовых: 3
Других типов: 3 ['amenities', 'first_review', 'last_review']

Пример текстового признака (description):
Simple, cozy and silent room in a lived house (me and eventual other guest or a friend); location of the house is central and offers great food, museums and green places.<br />The room is equipped with secretary, chair, closet, night table, a double bed. Furniture could be different from pictures. No locker. Kitchen and bathroom are shared and must be kept clean.<br />I request you to read the rul


## 3. Целевая переменная и пропуски

In [4]:
df["price"] = df["price"].str.replace(r"[$,]", "", regex=True).astype(float)

print(f"Пропуски в признаках: {df[features].isna().values.mean()*100:.1f}% ячеек")
print(f"Пропуски во всей таблице: {df.isna().values.mean()*100:.1f}% ячеек")
print(f"Строк хотя бы с одним пропуском: {df[features].isna().any(axis=1).mean()*100:.1f}%")
print(f"Пропуски в целевой price: {df['price'].isna().mean()*100:.1f}%")
print()
print(df[features].isna().mean().sort_values(ascending=False).head(10).mul(100).round(1))

Пропуски в признаках: 4.2% ячеек
Пропуски во всей таблице: 17.8% ячеек
Строк хотя бы с одним пропуском: 77.4%
Пропуски в целевой price: 8.2%

host_about                     54.9
host_location                  33.7
bedrooms                       17.8
bathrooms                      14.6
beds                           11.6
review_scores_checkin           9.9
review_scores_location          9.9
review_scores_communication     9.9
review_scores_value             9.9
reviews_per_month               9.9
dtype: float64


## 4. Выбросы (правило 1.5·IQR)

In [5]:
p = df["price"].dropna()
q1, q3 = p.quantile([0.25, 0.75]); iqr = q3 - q1
price_out = ((p < q1 - 1.5*iqr) | (p > q3 + 1.5*iqr)).mean() * 100
print(p.describe(percentiles=[.5, .9, .99]).round(1))
print(f"\nВыбросы по цене: {price_out:.1f}% (граница {q3 + 1.5*iqr:.0f} EUR, максимум {p.max():.0f} EUR)")

X = df[num_cols]
Q1, Q3 = X.quantile(0.25), X.quantile(0.75); I = Q3 - Q1
out = (X < Q1 - 1.5*I) | (X > Q3 + 1.5*I)
print(f"Выбросы среди числовых значений: {out.values.sum() / X.notna().values.sum() * 100:.1f}%")

count     4544.0
mean       175.7
std        387.6
min         15.7
50%        128.5
90%        256.5
99%        845.2
max      10000.0
Name: price, dtype: float64

Выбросы по цене: 6.8% (граница 298 EUR, максимум 10000 EUR)
Выбросы среди числовых значений: 5.2%
